# LLM fold pipeline — template, not yet run

**Author:** SF — **Branch:** `foldingstrategiesv2` — **Scope:** a **template**. `CONFIG
["run_training"]` is `False`; the cell that would actually call an LLM is gated behind it
and calls `call_llm_stub` (`scripts/llm_pipeline_utils.py`), which deliberately raises
`NotImplementedError` — there is no real provider wired up yet. This notebook is safe to
"Run All" today — it builds prompts and inspects the splits, and cannot call an API by
construction.

## Why a prompted LLM (see the full two-model writeup in this session's reply for the
other candidate and the comparison between them)

Every model tried so far — `LogisticRegression`, `RandomForestClassifier`,
`HistGradientBoostingClassifier` — is a supervised classifier trained on the *other* 5 use
cases' embeddings. None of them has any mechanism to read a *new* use case's own written
brief. That's not a detail; `papers_combined.parquet` broadcasts each use case's actual
`objective`, `terms_must_include`/`terms_nice_to_have`/`terms_exclude`, `trl_min`/`trl_max`,
and `decision_rules` onto every one of its rows — real, human-written inclusion/exclusion
criteria that every classifier tried so far has simply never been given as input. A
prompted LLM is the one architecture on the table that can consume that text directly, in
context, for a use case it has zero labelled training rows for — which is exactly the "0
labels, brand-new use case" rung of `reports/wf_ensemble_report.md` §4's own adaptation
ladder, the one rung nothing else tried has anything to offer.

**This is a genuinely different kind of model, not a better version of what's already
been tried** — no embedding, no scaling, no tree splits. The transformation that makes a
row "valid input" here is **prompt construction**, not `StandardScaler`. The one thing
that carries over unchanged from the `LogisticRegression`/LightGBM pipelines is the
split-then-fit discipline: few-shot demonstration examples are the prompt-based analogue of
a fitted parameter, and `select_few_shot_examples` (§4) draws them from the training fold
only, for the same reason a scaler is fit on the training fold only.

## What this notebook does

Combines both fold strategies from `notebooks/pipelines/` — pooled/generalized and
Leave-One-Use-Case-Out — against a prompted-LLM classifier instead of a
`scikit-learn` model. Reuses `scripts/fold_pipeline_utils.py`'s split/leakage/metric
helpers unchanged (they're model-agnostic); uses the new `scripts/llm_pipeline_utils.py`
for prompt construction, few-shot selection, and response parsing.

**Not executed in this session, and cannot be by construction** — `call_llm_stub` raises
`NotImplementedError`. See §7 for what to wire up before flipping `run_training` to
`True`.


In [ ]:
import sys
from pathlib import Path

SCRIPTS_DIR = Path("../../scripts").resolve()
sys.path.insert(0, str(SCRIPTS_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold

from fold_pipeline_utils import bootstrap_auc_ci, classification_metrics, prepare_dataset, ranking_metrics, validate_schema
from llm_pipeline_utils import build_prompt, build_use_case_brief, call_llm_stub, parse_llm_response, select_few_shot_examples

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## CONFIG

Same shape as the other `pipelines/` notebooks' CONFIG for the columns the split/leakage
logic needs (`use_case_col`, `group_col`, `title_col`) — those don't change with the
model. New/different here:

- `expand_embedding: False` — this model never touches the embedding column at all.
- `use_case_brief_cols` — the actual broadcast use-case-brief columns in
  `papers_combined.parquet` today (`problem_statement`, `objective`,
  `terms_must_include`, ...) — verified against the real file, not guessed. **Re-check
  this list against the real engineered dataset's schema** if/when the feature-engineering
  track restructures or renames any of these.
- `text_cols` — `(title_col, abstract_col)`, what goes into the prompt as the paper itself.
- `n_few_shot` — 0 for pure zero-shot; > 0 draws that many labelled demonstration examples
  from the training fold only (§4).
- `llm_model_name` — pinned explicitly, on purpose (`reports/wf_ensemble_report.md` §0's
  `relevance_score` lesson: never let a scored signal depend on an unpinned, driftable
  model/prompt version).


In [ ]:
CONFIG = {
    "data_path": "../../data/processed/papers_combined.parquet",
    # target
    "target_col": "triage_label",
    "positive_label": "positive",
    "negative_labels": ["negative"],
    # columns used by the split/leakage logic, not as model features - unchanged from the
    # other pipelines/ notebooks
    "use_case_col": "use_case_key",
    "group_col": "first_author",
    "group_source_col": "authors",
    "title_col": "title",
    # this model doesn't use the embedding or engineered numeric/categorical features at
    # all - it reads raw text - so these two lists stay empty and expand_embedding is off
    "embedding_col": "embedding",
    "expand_embedding": False,
    "numeric_feature_cols": [],
    "categorical_feature_cols": [],
    # LLM-specific: the use case's own written brief, broadcast onto every row of
    # papers_combined.parquet - verified against the real file's actual columns
    "use_case_brief_cols": [
        "problem_statement", "objective", "domain_industry", "domain_application",
        "domain_technology_focus", "terms_must_include", "terms_nice_to_have",
        "terms_exclude", "performance_criteria", "trl_min", "trl_max",
        "constraints_scale", "constraints_cost", "decision_must_have",
        "decision_nice_to_have", "decision_exclusions", "decision_rules",
    ],
    "text_cols": ("title", "abstract"),
    "n_few_shot": 4,   # 0 = pure zero-shot; drawn from the training fold only, see §4
    "llm_model_name": "claude-sonnet-5",   # pin this - see §0's note on relevance_score's drift lesson
    # split sizes - identical role to the other pipelines/ notebooks
    "n_splits_outer": 5,
    "n_splits_inner": 5,
    "random_state": 0,
    # TEMPLATE SWITCH - leave False. call_llm_stub raises NotImplementedError regardless,
    # so this can't actually call anything even if left True by mistake - belt and braces.
    "run_training": False,
}


## 0. Load data + schema check — no feature-engineering stand-in needed here

Unlike the `LogisticRegression`/LightGBM pipelines, this notebook doesn't need §0's
temporary `citation_velocity` stopgap — this model doesn't consume engineered numeric
features at all, only raw text + the use-case brief, both already present in
`papers_combined.parquet` today.


In [ ]:
df_raw = pd.read_parquet(CONFIG["data_path"])
print(f"Loaded {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns from {CONFIG['data_path']}")

validate_schema(df_raw, CONFIG)
df = prepare_dataset(df_raw, CONFIG)
USE_CASES = sorted(df[CONFIG["use_case_col"]].unique().tolist())
print(f"{len(df):,} labelled rows (positive/negative only) across {len(USE_CASES)} use cases")
print(f"embedding expansion skipped (expand_embedding=False) - "
      f"{len(CONFIG['_embedding_feature_cols'])} embedding feature columns")


## 1. Duplicate-row leakage check

Same check as every other `pipelines/` notebook — model-agnostic.


In [ ]:
title_norm = df[CONFIG["title_col"]].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
n_dup_titles = title_norm.duplicated(keep=False).sum()
print(f"{n_dup_titles} rows share a title with at least one other row in the pooled dataset "
      f"(out of {len(df):,})")


## 2. Where the split needs to happen, and why — restated for a prompted model

The principle is identical to every other notebook in `pipelines/`: nothing below this
line may use validation/holdout rows to build anything the model conditions on. For a
prompted LLM, the thing that would leak isn't a fitted scaler — it's **which rows end up
as few-shot demonstrations in the prompt**. `select_few_shot_examples` (§4) is written to
only ever receive a fold/rotation's *training* rows; passing it `train_pool`/`dev_pool`
(not the whole `df`) is what keeps this leakage-safe, exactly as
`StandardScaler().fit(X_train)` (not `X`) does for the other pipelines.

### Part A — generalized (pooled) split


In [ ]:
strat_key_outer = df[CONFIG["use_case_col"]].astype(str) + "__" + df["y"].astype(str)
outer_cv = StratifiedGroupKFold(
    n_splits=CONFIG["n_splits_outer"], shuffle=True, random_state=CONFIG["random_state"]
)
outer_splits = list(outer_cv.split(df, strat_key_outer, groups=df[CONFIG["group_col"]]))
train_pool_idx, final_holdout_idx = outer_splits[0]

train_pool_a = df.iloc[train_pool_idx].reset_index(drop=True)
final_holdout_a = df.iloc[final_holdout_idx].reset_index(drop=True)
print(f"Part A: train_pool {len(train_pool_a):,} rows | final_holdout {len(final_holdout_a):,} rows")


### Part B — Leave-One-Use-Case-Out rotation

No inner CV needed here the way the sklearn-model pipelines use it — there's no fitted
statistic to validate a hyperparameter choice against (a prompt has no coefficients).
`dev_pool` still serves as the pool `select_few_shot_examples` draws from per rotation.


In [ ]:
print(f"Part B: {len(USE_CASES)} rotations, one per use case: {USE_CASES}")


## 3. Inspecting a prompt, without calling anything

Building and reading one real prompt end-to-end, entirely offline — this is the
"transformation, to be valid for this model" step, the direct counterpart to
`build_preprocessor`'s `ColumnTransformer` in the sklearn pipelines. `few_shot` is drawn
from `train_pool_a` only (see §2's leakage note), never from `final_holdout_a`.


In [ ]:
sample_row = final_holdout_a.iloc[0]
brief_text = build_use_case_brief(sample_row, CONFIG)
few_shot = select_few_shot_examples(train_pool_a, CONFIG, random_state=CONFIG["random_state"])
prompt = build_prompt(sample_row, brief_text, few_shot, CONFIG)

print(f"Use-case brief for this row's use case ({sample_row[CONFIG['use_case_col']]}):\n")
print(brief_text)
print(f"\n--- {len(few_shot)} few-shot example(s) drawn from train_pool_a only ---")
print(f"\n--- Full prompt ({len(prompt)} chars) ---\n")
print(prompt[:2000] + ("..." if len(prompt) > 2000 else ""))


## 4. Scoring loop (gated) — Part A and Part B

The loop this notebook would run once wired to a real provider: for each row, build its
prompt (few-shot examples drawn from that split's training rows only), call the LLM, parse
the response into `{"pred", "proba"}`, then score with the same
`classification_metrics`/`ranking_metrics`/`bootstrap_auc_ci` every other pipeline notebook
uses. `call_llm_stub` raises `NotImplementedError` — this cannot silently produce fabricated
results even if `run_training` were accidentally left `True`.


In [ ]:
def score_llm(df_to_score, few_shot_pool, config):
    """Run the LLM classifier over every row in df_to_score, with few-shot examples
    drawn once from few_shot_pool (a training-fold-only pool - never the pool being
    scored). Returns arrays of (y_true, proba, pred) for classification_metrics/
    ranking_metrics. Calls call_llm_stub per row, which raises NotImplementedError until
    a real provider is wired up."""
    few_shot = select_few_shot_examples(few_shot_pool, config, random_state=config["random_state"])
    y_true, proba, pred = [], [], []
    for _, row in df_to_score.iterrows():
        brief_text = build_use_case_brief(row, config)
        prompt = build_prompt(row, brief_text, few_shot, config)
        raw_response = call_llm_stub(prompt, config)   # raises NotImplementedError today
        parsed = parse_llm_response(raw_response)
        y_true.append(row["y"])
        proba.append(parsed["proba"])
        pred.append(parsed["pred"])
    return np.array(y_true), np.array(proba), np.array(pred)


if CONFIG["run_training"]:
    # Part A
    y_true_a, proba_a, pred_a = score_llm(final_holdout_a, train_pool_a, CONFIG)
    holdout_metrics_a = classification_metrics(y_true_a, proba_a, pred_a)
    holdout_metrics_a.update(ranking_metrics(y_true_a, proba_a))
    print("Part A holdout metrics")
    display(pd.Series(holdout_metrics_a).round(3))

    # Part B
    logo_rows = []
    for holdout_uc in USE_CASES:
        dev_pool = df[df[CONFIG["use_case_col"]] != holdout_uc].reset_index(drop=True)
        holdout = df[df[CONFIG["use_case_col"]] == holdout_uc].reset_index(drop=True)
        y_true_r, proba_r, pred_r = score_llm(holdout, dev_pool, CONFIG)
        m = classification_metrics(y_true_r, proba_r, pred_r)
        m["holdout_use_case"] = holdout_uc
        logo_rows.append(m)
    logo_results = pd.DataFrame(logo_rows)
    print("Part B: LOGO rotation summary")
    display(logo_results)
else:
    print("Skipped: CONFIG['run_training'] is False (and call_llm_stub would raise "
          "NotImplementedError even if it weren't).")


## 5. What this template does and doesn't decide for you

- **Zero-shot vs. few-shot** (`CONFIG["n_few_shot"]`) is left as a config knob, not a
  decision made here — worth testing both once real calls are wired up, since a new use
  case with genuinely 0 labelled examples (the actual "day one" scenario this whole
  approach is aimed at) can only ever run zero-shot.
- **Which LLM** (`CONFIG["llm_model_name"]`) is a placeholder value, not a recommendation
  fixed by this notebook — pick and pin one deliberately when ready.
- **This is not a replacement for the evaluation discipline already built** — once wired
  up, this notebook's Part B numbers should be compared against `sf_logo_fold_pipeline.ipynb`'s
  and `sf_lightgbm_fold_pipeline.ipynb`'s on the exact same rotations, the same way every
  other model in this project has been compared, not assumed to be better because the
  underlying idea targets the right problem.

## 6. Checklist — dataset swap

If the feature-engineering track changes the use-case-brief columns' names or the
title/abstract columns:

1. Update `CONFIG["use_case_brief_cols"]` and `CONFIG["text_cols"]` to match.
2. Re-run `validate_schema` (§0) — it fails loudly on any mismatch.
3. Nothing else in this notebook depends on the engineered numeric/categorical feature
   columns the other two `pipelines/` notebooks care about — this model never reads them.

## 7. Checklist — before flipping `run_training` to `True`

1. Implement a real call in `call_llm_stub` (`scripts/llm_pipeline_utils.py`) — pin the
   provider, model version, and this notebook's exact prompt-template version together;
   log all three alongside every stored response.
2. Decide and test `CONFIG["n_few_shot"]` (§5).
3. Budget for cost/latency — this loops one API call per row, per stage, per rotation;
   consider caching responses keyed on `(paper_id, use_case_key, prompt_version)` before
   running the full LOGO sweep.
4. Only then set `CONFIG["run_training"] = True`.
